# Hands-On: Reinforcement Learning for a Vision-Language-Action Robot Policy with RLinf on AMD Radeon™ GPUs

In this workshop you will fine-tune a **Vision-Language-Action (VLA)** robot policy with **reinforcement learning (RL)** in a physics simulator, end to end, on **4 AMD Radeon™ PRO W7900D GPUs**.

| Component | What we use |
| --- | --- |
| RL framework | [RLinf](https://github.com/RLinf/RLinf), a scalable RL infrastructure for embodied and agentic AI |
| Policy | **Lingbot-VLA** (4.2 B parameters), already supervised-fine-tuned (SFT) on RoboTwin tasks |
| Simulator | [RoboTwin 2.0](https://github.com/RoboTwin-Platform/RoboTwin) (SAPIEN physics + Vulkan rendering), dual-arm ALOHA robot |
| Task | `place_shoe`: pick up a shoe and place it on the target mat |
| Algorithm | **GRPO** (Group Relative Policy Optimization) with a task-success reward |
| Hardware / software | 4 × Radeon PRO W7900D (gfx1100, 48 GB), ROCm 7.2, PyTorch 2.11 |

**What you will do in the next ~40 minutes**

1. Check the GPUs and the software environment in your container (~3 min).
2. Walk through the key training parameters (~5 min).
3. Launch a **40-step GRPO training run** that evaluates the policy on held-out scenes **every 10 steps**. The first evaluation result arrives **about 16–18 minutes** after launch (~55–75 s per training step), then a new one about every 13 minutes, so you will see two to three evaluations during the workshop. The whole run takes about 55 minutes.
4. Watch training progress live, read the metrics, and play the evaluation videos of each evaluation.

> This notebook is a *demo-sized* run: we shrink the number of parallel environments so that 10 RL steps fit into the workshop. The last section shows the full-size settings.

## Background: RL Fine-Tuning of a VLA Policy

A VLA policy maps **camera images + robot state + a language instruction** to robot **actions**. Lingbot-VLA predicts an *action chunk*: 50 future joint-space actions for the two arms at a time. The simulator executes the chunk, renders new camera images, and the policy predicts the next chunk. A `place_shoe` episode is at most 400 simulator steps, i.e. 8 chunks.

Supervised fine-tuning (SFT) teaches the policy to imitate demonstrations. **RL fine-tuning** then lets the policy practise in the simulator and reinforces the behaviour that actually completes the task.

### GRPO for robot policies

For each initial scene (object poses, lighting, textures) RLinf runs a **group** of $G$ episodes with the current policy and gives each episode a reward $R_i$, here simply whether the task succeeded:

$$R_i = \mathbb{1}[\text{episode } i \text{ succeeded}], \qquad i = 1, \dots, G$$

Instead of training a value network (critic), GRPO uses the group itself as the baseline:

$$\hat A_i = \frac{R_i - \operatorname{mean}(R_1, \dots, R_G)}{\operatorname{std}(R_1, \dots, R_G)}$$

Episodes that succeed where their siblings failed get a positive advantage; the policy is then updated with the clipped policy-gradient objective of PPO over the log-probabilities of the executed actions:

$$\mathcal{L}(\theta) = -\,\mathbb{E}\Big[\min\big(\rho_t \hat A,\ \operatorname{clip}(\rho_t, 1-\epsilon_\text{low}, 1+\epsilon_\text{high})\,\hat A\big)\Big], \qquad \rho_t = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_\text{old}}(a_t \mid s_t)}$$

The recipe uses an asymmetric clip range, $\epsilon_\text{low} = 0.2$ and $\epsilon_\text{high} = 0.28$ (`algorithm.clip_ratio_low` / `clip_ratio_high`).

If every episode in a group succeeds (or every one fails), all advantages are zero and the group teaches nothing. That is why a **strong SFT policy** (success rate ~90 %) still benefits from RL: the remaining failures are exactly what GRPO learns from.

In this workshop only the **action expert** of Lingbot-VLA (≈0.44 B of the 4.2 B parameters) is trained; the vision-language backbone stays frozen.

### How RLinf runs one training step

RLinf starts three groups of workers with [Ray](https://www.ray.io/) and places all of them on the same 4 GPUs (one worker of each kind per GPU):

```text
                ┌───────────────────────── one GPU (× 4) ─────────────────────────┐
                │                                                                  │
  RoboTwin  ◀──▶│  EnvWorker            MultiStepRolloutWorker     EmbodiedFSDPActor│
  scenes        │  (SAPIEN physics,     (policy inference:         (FSDP training  │
                │   Vulkan cameras)      action chunks + log-probs) of the policy)  │
                └──────────────────────────────────────────────────────────────────┘

  1. rollout   EnvWorker ⇄ RolloutWorker: run every environment for one episode (8 action chunks)
  2. advantage group-relative advantages from the success rewards
  3. update    Actor: policy-gradient update on the collected trajectories
  4. sync      Actor → RolloutWorker: send the updated weights for the next rollout
  5. eval      every N steps: run held-out scenes with the current policy (no training)
```

Workers that are idle in a phase move their model to CPU memory (**offload**), so the three models can share each 48 GB GPU.

## 1. Verify the AMD GPUs and the Software Environment

### Check the GPUs

`rocm-smi` lists the AMD GPUs visible inside this container with their temperature, power, memory use (`VRAM%`) and utilisation (`GPU%`):

In [ ]:
!rocm-smi

### Choose your GPUs

The run uses **4 GPUs**. `GPU_GROUP` selects them by index: keep `"0-3"` on a 4-GPU machine; on a machine with more GPUs, choose 4 consecutive free ones (for example `"4-7"`). The next cell also holds all paths; you normally do not need to change anything.

In [ ]:
import os
import subprocess
from pathlib import Path

# ---- the 4 GPUs to train on: "0-3" on a 4-GPU machine ----
GPU_GROUP = "0-3"

# ---- paths inside the workshop container (prepared by the organizers) ----
RLINF_ROOT    = "/workspace/RLinf"                     # RLinf source tree
VENV          = "/opt/venv/lingbotvla"                 # Python environment for Lingbot-VLA
ROBOTWIN_PATH = "/opt/robotwin"                        # RoboTwin checkout; assets in $ROBOTWIN_PATH/assets
MODEL_ROOT    = "/data/models"
POLICY_CKPT   = f"{MODEL_ROOT}/lingbot-vla-4b-posttrain-robotwin"   # SFT policy for RoboTwin
TOKENIZER     = f"{MODEL_ROOT}/Qwen2.5-VL-3B-Instruct"              # tokenizer / processor
LINGBOT_CFG   = f"{MODEL_ROOT}/lingbot-vla-4b"                      # Lingbot-VLA model config
WORK_DIR      = "/workspace/rlinf_workshop"            # logs, metrics and videos of your runs

RUN_NAME = f"lingbot_place_shoe_grpo_gpu{GPU_GROUP.replace('-', '_')}"
RUN_DIR  = f"{WORK_DIR}/{RUN_NAME}"

# export everything so the %%bash cells below can use it
for k, v in dict(GPU_GROUP=GPU_GROUP, RLINF_ROOT=RLINF_ROOT, VENV=VENV, ROBOTWIN_PATH=ROBOTWIN_PATH,
                 POLICY_CKPT=POLICY_CKPT, TOKENIZER=TOKENIZER, LINGBOT_CFG=LINGBOT_CFG,
                 WORK_DIR=WORK_DIR, RUN_NAME=RUN_NAME, RUN_DIR=RUN_DIR).items():
    os.environ[k] = v

ok = True
for name, path in [("RLinf", RLINF_ROOT), ("Python env", VENV), ("RoboTwin", ROBOTWIN_PATH),
                   ("RoboTwin assets", f"{ROBOTWIN_PATH}/assets"), ("policy checkpoint", POLICY_CKPT),
                   ("tokenizer", TOKENIZER), ("Lingbot-VLA config", LINGBOT_CFG)]:
    exists = Path(path).exists()
    ok &= exists
    print(f"{'OK     ' if exists else 'MISSING'} {name:20s} {path}")

first, last = (int(x) for x in GPU_GROUP.split("-"))
gpus = list(range(first, last + 1))
print(f"\nGPU group {GPU_GROUP} -> GPUs {gpus}")
csv = subprocess.run(["rocm-smi", "--showuse", "--showmemuse", "--csv"], capture_output=True, text=True).stdout
for line in csv.splitlines():
    if line.startswith("card") and int(line.split(",")[0][4:]) in gpus:
        card, use, vram = line.split(",")[:3]
        print(f"  {card}: GPU use {use:>3}%  VRAM {vram:>3}%" + ("   <-- busy?" if int(vram) > 5 else ""))
print("\nAll paths found." if ok else "\nSome paths are missing: ask an instructor before continuing.")

### Verify the Python environment

The container ships one Python environment per model under `/opt/venv`. Activate the Lingbot-VLA environment and check that PyTorch sees the AMD GPUs through ROCm/HIP, and that RLinf and the simulator import:

In [ ]:
%%bash
set -e
source "${VENV}/bin/activate"
cd "${RLINF_ROOT}"
export PYTHONPATH="${RLINF_ROOT}:${ROBOTWIN_PATH}"

(python - <<'PY'
import torch
print("PyTorch      :", torch.__version__, "| HIP", torch.version.hip)
print("GPUs visible :", torch.cuda.device_count(), "x", torch.cuda.get_device_name(0))
import sapien, rlinf, os
print("SAPIEN       :", sapien.__version__)
print("RLinf        :", os.path.dirname(rlinf.__file__))
from rlinf.envs.sim.robotwin.robotwin_env import RoboTwinEnv  # noqa: F401
print("RoboTwin env : import OK")
print("flash-attn backend:", "Triton" if os.environ.get("FLASH_ATTENTION_TRITON_AMD_ENABLE") == "TRUE" else "CK")
PY
) 2>&1 | grep -v "(null): No such file or directory"

## 2. The Training Configuration

RLinf configs are [Hydra](https://hydra.cc/) YAML files. We start from the recipe
`examples/embodiment/config/robotwin_place_shoe_grpo_lingbotvla.yaml` and override a few values on the command line. Print the parts of the recipe that matter here:

In [ ]:
%%bash
source "${VENV}/bin/activate"
python - <<'PY'
import os
from omegaconf import OmegaConf
cfg = OmegaConf.load(os.path.join(os.environ["RLINF_ROOT"], "examples/embodiment/config/robotwin_place_shoe_grpo_lingbotvla.yaml"))
show = {
    "cluster.component_placement": cfg.cluster.component_placement,
    "runner": {k: cfg.runner[k] for k in ("max_steps", "val_check_interval", "save_interval")},
    "algorithm": {k: cfg.algorithm[k] for k in ("adv_type", "group_size", "update_epoch", "clip_ratio_low", "clip_ratio_high")},
    "env.train": {k: cfg.env.train[k] for k in ("total_num_envs", "rollout_epoch", "max_episode_steps", "group_size")},
    "env.eval": {k: cfg.env.eval[k] for k in ("total_num_envs", "rollout_epoch", "max_episode_steps")},
    "rollout": {k: cfg.rollout[k] for k in ("backend", "pipeline_stage_num", "enable_offload")},
    "actor": {"micro_batch_size": cfg.actor.micro_batch_size, "global_batch_size": cfg.actor.global_batch_size,
              "optim.lr": cfg.actor.optim.lr, "enable_offload": cfg.actor.enable_offload},
}
print(OmegaConf.to_yaml(OmegaConf.create({k.replace(".", "/"): v for k, v in show.items()})))
PY

### Workshop overrides and why

The full recipe trains on 128 parallel environments with large batches; one step takes several minutes. For the workshop we keep the algorithm and shrink the batch so that every 10 steps, including an evaluation, take less than 20 minutes.

| Override | Recipe default | Workshop | Why |
| --- | --- | --- | --- |
| `cluster.component_placement` | `actor, env, rollout: 0-7` | `{actor: <group>, env: <group>, rollout: <group>}` | Use only your 4 GPUs; one worker of each kind per GPU |
| `env.train.total_num_envs` | 128 | **16** | 4 environments per GPU: step time grows with environments per GPU |
| `algorithm.group_size` | 8 | **4** | GRPO group size; with 16 environments this gives 4 groups per step |
| `env.train.rollout_epoch` | 8 | **1** | One episode per environment per step |
| `actor.global_batch_size` / `actor.micro_batch_size` | 1024 / 32 | **16 / 4** | All 16 trajectories form one global batch; `global_batch_size` must be divisible by `micro_batch_size × 4 GPUs` |
| `algorithm.update_epoch` | 2 | **1** | One pass over each batch |
| `env.eval.total_num_envs` | 128 | **32** | 32 held-out evaluation episodes |
| `runner.max_steps` / `runner.val_check_interval` | -1 / 20 | **40 / 10** | Train 40 steps, evaluate after steps 10, 20, 30 and 40 |
| `runner.save_interval` | 20 | **-1** | No checkpoints (each would be ~18 GB) |
| `env.eval.video_cfg.save_video` / `fps` | True / 30 | **True / 2** | Save evaluation videos; one frame per action chunk, played at 2 fps |
| `env.train.video_cfg.save_video` | True | **False** | Skip training videos |

Constraints RLinf checks at start-up:

- `total_num_envs / num_GPUs` must be divisible by `group_size` (each GPU holds whole groups),
- `global_batch_size` must be divisible by `micro_batch_size × num_GPUs`.

> **On AMD Radeon GPUs, keep `rollout.pipeline_stage_num: 1` (the default) and do not enable `runner.overlap_env_bootstrap`.** Both make Vulkan rendering and HIP compute run on the same GPU at the same time; on Radeon PRO W7900D this has caused GPU hangs (`HW Exception ... GPU Hang`, `vk::Device::waitForFences: ErrorDeviceLost`). They also save little time here: the step time is dominated by resetting the RoboTwin scenes.

## 3. Launch Training

The next cell starts training **in the background** and returns immediately; logs go to `$RUN_DIR/train.log` and metrics to `$RUN_DIR/metrics.log`. Re-running the cell while a run is active does nothing.

In [ ]:
%%bash
set -euo pipefail
mkdir -p "${RUN_DIR}"
LOG_FILE="${RUN_DIR}/train.log"
PID_FILE="${RUN_DIR}/train.pid"

# a process counts as running unless it is gone or a zombie (finished, not yet reaped)
is_running() { local s; s=$(ps -o stat= -p "$1" 2>/dev/null); [[ -n "${s}" && "${s}" != Z* ]]; }
if [[ -f "${PID_FILE}" ]] && is_running "$(cat "${PID_FILE}")"; then
    echo "Training is already running (PID $(cat "${PID_FILE}")). Log: ${LOG_FILE}"
    exit 0
fi

nohup setsid bash -s >"${LOG_FILE}" 2>&1 <<'TRAIN_SCRIPT' &
set -uo pipefail
source "${VENV}/bin/activate"
cd "${RLINF_ROOT}"
export PYTHONPATH="${RLINF_ROOT}:${ROBOTWIN_PATH}"
export REPO_PATH="${RLINF_ROOT}" EMBODIED_PATH="${RLINF_ROOT}/examples/embodiment"
export ROBOT_PLATFORM=ALOHA ROBOTWIN_PATH="${ROBOTWIN_PATH}"
export LINGBOT_VLA_PATH="${LINGBOT_VLA_PATH:-${VENV}/lingbot-vla}"
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl TOKENIZERS_PARALLELISM=false PYTHONUNBUFFERED=1 HYDRA_FULL_ERROR=1

echo "TRAIN_START $(date -u '+%F %T') UTC  GPU group ${GPU_GROUP}"
python examples/embodiment/train_embodied_agent.py \
    --config-path "${RLINF_ROOT}/examples/embodiment/config" \
    --config-name robotwin_place_shoe_grpo_lingbotvla \
    '~cluster.component_placement' \
    "+cluster.component_placement={actor:${GPU_GROUP},env:${GPU_GROUP},rollout:${GPU_GROUP}}" \
    runner.logger.log_path="${RUN_DIR}" \
    runner.max_steps=40 runner.val_check_interval=10 runner.save_interval=-1 \
    algorithm.group_size=4 algorithm.update_epoch=1 \
    env.train.total_num_envs=16 env.train.rollout_epoch=1 \
    env.eval.total_num_envs=32 \
    actor.global_batch_size=16 actor.micro_batch_size=4 \
    env.train.assets_path="${ROBOTWIN_PATH}" env.eval.assets_path="${ROBOTWIN_PATH}" \
    env.train.video_cfg.save_video=False \
    env.eval.video_cfg.save_video=True +env.eval.video_cfg.fps=2 \
    actor.model.model_path="${POLICY_CKPT}" \
    actor.model.tokenizer_path="${TOKENIZER}" \
    actor.model.lingbotvla.config_path="${LINGBOT_CFG}"
echo "TRAIN_EXIT=$?  $(date -u '+%F %T') UTC"
TRAIN_SCRIPT

echo $! > "${PID_FILE}"
echo "Training started in the background."
echo "  PID : $(cat "${PID_FILE}")"
echo "  Log : ${LOG_FILE}"

### Watch training live

This cell refreshes every 15 seconds while training runs. It shows the time since launch, the current phase, one row per finished step, and the success rates.

**It stops by itself as soon as a new evaluation result arrives**, so that you can run the cells below to plot the curves and play the new evaluation videos. Run it again to keep watching until the next evaluation. You can also stop it any time with the **Stop (■)** button. Stopping this cell does **not** stop training.

Expected timeline on 4 × W7900D:

| Phase | Time |
| --- | --- |
| Start Ray, load the policy into 4 rollout + 4 actor workers, build 16 + 32 environments | ~2.5 min |
| Step 1 (first inference of each worker and the first full weight sync) | ~130 s |
| Every other step | ~55–75 s |
| Evaluation after steps 10, 20, 30, 40 (32 episodes each) | ~90 s; up to ~3 min if other jobs load the machine's CPUs (the simulator is CPU-bound) |
| **First evaluation result (after step 10)** | **~16–18 min after launch** |
| Each further 10 steps + evaluation | ~13 min |
| Whole 40-step run | ~55 min |

In [ ]:
import calendar, re, time
from IPython.display import clear_output, display

ANSI = re.compile(r"\x1b\[[0-9;]*m")
NOISE = re.compile(r"svulkan2|denoiser|cudaError|curobo|Curobo|pytorch3d|planner\.py|Exception traceback|Traceback \(most recent|"
                   r"No such file or directory|GetPrototype|FlexiblePlacement|Hydra14Migration|is unstable|^\s*$|"
                   r"fsdp_model_manager|tie embeddings|slow image processor|oneDNN|warnings\.warn|UserWarning")

def parse_metrics(path):
    # Parse RLinf's console metric tables into one dict per step.
    try:
        txt = ANSI.sub("", Path(path).read_text(errors="ignore"))
    except FileNotFoundError:
        return []
    rows = []
    for block in txt.split("Metric Table")[1:]:
        m = re.search(r"Global Step:\s+(\d+)/(\d+)", block)
        if not m:
            continue
        row, section = {"step": int(m.group(1)), "of": int(m.group(2))}, ""
        for line in block.splitlines():
            h = re.search(r"├─+\s*([A-Za-z/ ]+?)\s*─+┤", line)
            if h:
                section = h.group(1).strip()
                continue
            for k, v in re.findall(r"([A-Za-z_/]+)=([-0-9.e+]+)", line):
                try:
                    row[f"{section}:{k}"] = float(v)
                except ValueError:
                    pass
        rows.append(row)
    return rows

def training_status():
    pid_file, log_file = Path(RUN_DIR, "train.pid"), Path(RUN_DIR, "train.log")
    log = ANSI.sub("", log_file.read_text(errors="ignore")) if log_file.exists() else ""
    m = re.search(r"TRAIN_EXIT=(\d+)", log)
    if m:
        return ("finished" if m.group(1) == "0" else f"FAILED (exit {m.group(1)})"), log
    stat = subprocess.run(["ps", "-o", "stat=", "-p", pid_file.read_text().strip()],
                          capture_output=True, text=True).stdout.strip() if pid_file.exists() else ""
    alive = bool(stat) and not stat.startswith("Z")  # a zombie has finished but was not reaped yet
    return ("running" if alive else "not running"), log

def current_phase(log):
    for line in reversed(log.splitlines()[-400:]):
        if "Generating Rollout Epochs" in line:
            return "rollout: " + line.split("Generating Rollout Epochs")[-1].strip()[:60]
        if "evaluate" in line.lower() and "Error" not in line:
            return "evaluation"
        if not NOISE.search(line):
            return line.strip()[-110:]
    return "starting"

def since_launch(log):
    m = re.search(r"TRAIN_START (\S+ \S+) UTC", log)
    if not m:
        return "-"
    start = calendar.timegm(time.strptime(m.group(1), "%Y-%m-%d %H:%M:%S"))  # the log time is UTC
    return time.strftime("%M:%S" if time.time() - start < 3600 else "%H:%M:%S", time.gmtime(time.time() - start))

def show(rows, status, log):
    clear_output(wait=True)
    done = rows[-1]["step"] if rows else 0
    total = rows[-1]["of"] if rows else 40
    print(f"[{time.strftime('%H:%M:%S')}] status: {status} | since launch {since_launch(log)} | steps done {done}/{total}")
    print(f"phase : {current_phase(log)}\n")
    print(f"{'step':>4} {'step s':>7} {'rollout s':>9} {'actor s':>7} {'eval s':>6} {'train success':>13} {'EVAL success':>12}")
    for r in rows:
        ev = r.get("Evaluation:success_once")
        print(f"{r['step']:>4} {r.get('Time:step', 0):>7.0f} {r.get('Time:generate_rollouts', 0):>9.0f} "
              f"{r.get('Time:actor_training', 0):>7.0f} {r.get('Time:eval', 0):>6.0f} "
              f"{r.get('Environment:success_once', 0):>13.3f} {('%.3f' % ev) if ev is not None else '':>12}")

def eval_steps(rows):
    return [r["step"] for r in rows if "Evaluation:success_once" in r]

seen = eval_steps(parse_metrics(Path(RUN_DIR, "metrics.log")))  # evaluations already shown before this run of the cell
try:
    while True:
        status, log = training_status()
        rows = parse_metrics(Path(RUN_DIR, "metrics.log"))
        show(rows, status, log)
        new = [s for s in eval_steps(rows) if s not in seen]
        if new:
            print(f"\nNew evaluation after step {new[-1]}: run the plot and video cells below, "
                  "then re-run this cell to keep watching.")
            break
        if status != "running":
            break
        time.sleep(15)
except KeyboardInterrupt:
    print("\nStopped watching (training keeps running in the background).")

> **Tip.** To see the raw log, run the next cell. It prints the last 40 lines of `train.log` without the known-harmless noise listed in *Troubleshooting*.

In [ ]:
_, log = training_status()
print("\n".join([l for l in log.splitlines() if not NOISE.search(l)][-40:]))

## 4. Understand the Metrics

RLinf prints a metric table after every step (also saved in `metrics.log` and as TensorBoard events under `$RUN_DIR/tensorboard`). The most useful entries:

**Environment and evaluation (task performance)**

| Metric | Meaning |
| --- | --- |
| `Environment:success_once` | Fraction of this step's training episodes in which the task succeeded at least once |
| `Environment:num_trajectories` | Training episodes collected this step (16) |
| `Evaluation:success_once` | Success rate on the held-out evaluation scenes, **the headline number** |
| `Evaluation:success_at_end` | Fraction of evaluation episodes still in the success state at the last step |
| `Evaluation:episode_len` | Episode length in simulator steps (400 = the time limit) |

**Rollout and training (RL signal)**

| Metric | Meaning |
| --- | --- |
| `Rollout:advantages_mean/min/max` | Group-relative advantages; all zeros means every group was all-success or all-failure |
| `Training/Actor:actor/policy_loss` | Clipped policy-gradient loss |
| `Training/Actor:actor/approx_kl`, `actor/clip_fraction` | How far one update moved the policy; small values mean a stable update |
| `Training/Actor:actor/grad_norm` | Gradient norm before clipping |
| `Training/Actor:actor/lr` | Learning rate |

**Time (seconds per step)**

| Metric | Meaning |
| --- | --- |
| `Time:step` | Whole step |
| `Time:generate_rollouts` | Rollout phase: scene reset (`env/env/bootstrap_step`), simulation and policy inference (`rollout/predict`) |
| `Time:actor_training` | Policy update |
| `Time:sync_weights` | Sending the new weights from the actor to the rollout workers |
| `Time:eval` | Evaluation (only on evaluation steps) |

With a strong SFT policy the success rate starts around 0.9, so 10 small steps are about seeing the full RL loop work, not about large gains. Note how noisy `Environment:success_once` is with only 16 episodes per step.

Plot the curves (run any time, also while training):

In [ ]:
import matplotlib.pyplot as plt

rows = parse_metrics(Path(RUN_DIR, "metrics.log"))
if not rows:
    print("No finished steps yet.")
else:
    steps = [r["step"] for r in rows]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    a1.plot(steps, [r.get("Environment:success_once", 0) for r in rows], "o-", label="train success (this step)")
    ev = [(r["step"], r["Evaluation:success_once"]) for r in rows if "Evaluation:success_once" in r]
    if ev:
        a1.plot(*zip(*ev), "r*", markersize=16, label="eval success")
    a1.set(xlabel="step", ylabel="success rate", ylim=(0, 1.05), title="Task success")
    a1.legend(loc="lower right")
    parts = [("rollout", "Time:generate_rollouts"), ("actor update", "Time:actor_training"),
             ("weight sync", "Time:sync_weights"), ("eval", "Time:eval")]
    bottom = [0.0] * len(rows)
    for label, key in parts:
        vals = [r.get(key, 0.0) for r in rows]
        a2.bar(steps, vals, bottom=bottom, label=label)
        bottom = [b + v for b, v in zip(bottom, vals)]
    a2.set(xlabel="step", ylabel="seconds", title="Where the time goes")
    a2.legend()
    plt.tight_layout()
    plt.show()

## 5. Evaluation Results and Videos

After steps 10, 20, 30 and 40, RLinf evaluates the policy on 32 held-out scenes and writes one video per GPU worker and evaluation to `$RUN_DIR/video/eval/seed_<worker seed>/<n>.mp4`, where `<n>` = 0, 1, 2, 3 is the first, second, ... evaluation. Each video tiles that worker's 8 evaluation environments; the overlay shows each episode's reward and termination flag. One frame is recorded per 50-action chunk, so a 400-step episode is about 10 frames; the very last frame already shows the scenes reset for the next episode.

In [ ]:
from IPython.display import HTML
import base64

EVAL_INDEX = -1   # which evaluation to show: -1 = latest, 0 = after step 10, 1 = after step 20, ...

rows = parse_metrics(Path(RUN_DIR, "metrics.log"))
evals = [r for r in rows if "Evaluation:success_once" in r]
if not evals:
    print("No evaluation yet: the first one runs after step 10.")
else:
    print(f"{'after step':>10} {'success':>8} {'episodes':>9} {'success_at_end':>15}")
    for r in evals:
        n = int(r.get("Evaluation:num_trajectories", 0))
        print(f"{r['step']:>10} {r['Evaluation:success_once']:>8.3f} {round(r['Evaluation:success_once'] * n):>4}/{n:<4} "
              f"{r.get('Evaluation:success_at_end', 0):>15.3f}")
    idx = EVAL_INDEX % len(evals)
    videos = sorted(Path(RUN_DIR, "video", "eval").glob(f"seed_*/{idx}.mp4"))
    print(f"\nVideos of the evaluation after step {evals[idx]['step']}: {len(videos)} file(s)")
    display(HTML("".join(
        f"<div style='display:inline-block;margin:4px;text-align:center'><video width='420' controls loop autoplay muted "
        f"src='data:video/mp4;base64,{base64.b64encode(v.read_bytes()).decode()}'></video><br>{v.parent.name}/{v.name}</div>"
        for v in videos)))

## 6. Stop Training and Clean Up

The run stops by itself after the evaluation at step 40 (about 55 minutes after launch). **If it is still running at the end of the workshop, run this cell to free the GPUs.** You can also use it to stop a run early. It interrupts the training process group, waits for Ray to shut down, and then stops any Ray processes left in this container.

In [ ]:
%%bash
PID_FILE="${RUN_DIR}/train.pid"
is_running() { local s; s=$(ps -o stat= -p "$1" 2>/dev/null); [[ -n "${s}" && "${s}" != Z* ]]; }
if [[ -f "${PID_FILE}" ]] && is_running "$(cat "${PID_FILE}")"; then
    PID=$(cat "${PID_FILE}")
    echo "Stopping training (process group ${PID}) ..."
    kill -INT -- "-${PID}" 2>/dev/null || true
    for i in $(seq 1 60); do is_running "${PID}" || break; sleep 1; done
    if is_running "${PID}"; then echo "Still running, sending SIGTERM"; kill -TERM -- "-${PID}" 2>/dev/null || true; sleep 10; fi
    source "${VENV}/bin/activate" && ray stop >/dev/null 2>&1 || true
    echo "Stopped."
else
    echo "No training process is running."
fi
first=${GPU_GROUP%-*}; last=${GPU_GROUP#*-}
echo "GPU group ${GPU_GROUP}:"
rocm-smi --showuse --showmemuse --csv | awk -F, -v a="${first}" -v b="${last}" \
    '/^card/ {n = substr($1, 5) + 0; if (n >= a && n <= b) printf "  %s: GPU use %s%%  VRAM %s%%\n", $1, $2, $3}'

## 7. Going Further

**Run a full-size configuration.** Without the workshop overrides the recipe uses 128 environments, 8 rollout epochs and a global batch of 1024. For reference, RLinf's place_shoe runs with 128 environments, `env.train.rollout_epoch=2` and `actor.global_batch_size=256` on 8 × W7900D took about 6.5 minutes per step (66 minutes per 10 steps including a 128-episode evaluation), and the evaluation success rate averaged 0.925 over 100 steps.

**Rules of thumb for sizing a run on Radeon GPUs**

- Step time is set mainly by the number of environments per GPU: 4 per GPU ≈ 55–75 s, 8 ≈ 115 s, 16 ≈ 185 s per step (one rollout epoch). Adding GPUs at a fixed number of environments per GPU keeps the step time and increases the batch.
- Most of the rollout time is resetting the RoboTwin scenes and simulating, not policy inference.
- Keep `rollout.pipeline_stage_num: 1` and leave `runner.overlap_env_bootstrap` off (see section 2).
- One checkpoint of this setup is ~18 GB (`runner.save_interval`).

**Other recipes in RLinf for RoboTwin**: OpenVLA-OFT + GRPO/PPO, π₀ / π₀.₅ + PPO, Lingbot-VLA + GRPO on other tasks. See the [RLinf RoboTwin guide](https://rlinf.readthedocs.io/en/latest/rst_source/examples/embodied/robotwin.html).

## Troubleshooting

**Harmless log lines on AMD GPUs**

| Log line | Why it is harmless |
| --- | --- |
| `[svulkan2] [error] CUDA Error: cudaErrorInsufficientDriver` / `Failed to initialize denoiser` | SAPIEN probes for an NVIDIA denoiser; rendering uses Vulkan and works without it |
| `ModuleNotFoundError: No module named 'curobo'` (from `planner.py`) | The CUDA-only cuRobo planner is not installed; RoboTwin uses the `mplib` planner |
| `Objects is unstable in seed(...), trying new seed` | RoboTwin rejects a randomly generated scene and samples another one |
| `'MessageFactory' object has no attribute 'GetPrototype'` | protobuf version mismatch reported when a dependency imports TensorFlow; no effect |

**Problems**

| Symptom | Fix |
| --- | --- |
| A GPU in your group shows memory in use before you start | Another process is using it: choose other GPUs or ask an instructor |
| `AssertionError: ... must be divisible by ...` at start-up | See the two divisibility rules in section 2 |
| The run fails with `HW Exception ... GPU Hang` or `ErrorDeviceLost` | Make sure `rollout.pipeline_stage_num` is 1 and `runner.overlap_env_bootstrap` is not set; then relaunch |
| Host out of memory | One run peaks at ~170 GB of host RAM; do not start a second run in the same container |

---
## Appendix: Workshop Setup (for organizers)

Each participant gets one container with **4 GPUs** (for example one 4 × W7900D cloud instance per participant).

JupyterLab is not part of the RLinf image. Add it once in a small derived image, before the workshop (many containers installing it at the same time can hit PyPI-mirror rate limits):

```dockerfile
FROM <rlinf-rocm-robotwin-image>
RUN uv pip install --python /opt/venv/lingbotvla/bin/python jupyterlab ipykernel
```

```bash
docker build -t rlinf-ws:latest .

docker run -d --name rlinf-ws \
  --device /dev/kfd --device /dev/dri --group-add video --group-add render \
  --init --security-opt seccomp=unconfined --ipc=host --shm-size 64g -p 8888:8888 \
  -v /path/to/RLinf:/workspace/RLinf \
  -v /path/to/RoboTwin:/opt/robotwin -v /path/to/robotwin-assets:/opt/robotwin/assets \
  -v /path/to/models:/data/models:ro \
  -v /path/to/rlinf_lingbot_place_shoe_grpo_workshop.ipynb:/workspace/rlinf_lingbot_place_shoe_grpo_workshop.ipynb \
  rlinf-ws:latest \
  /opt/venv/lingbotvla/bin/jupyter lab --notebook-dir=/workspace --ip=0.0.0.0 --port=8888 \
      --allow-root --no-browser --ServerApp.token=<token>
```

Needed in `/data/models`: `lingbot-vla-4b-posttrain-robotwin` (SFT policy, ~16 GB), `Qwen2.5-VL-3B-Instruct` (tokenizer/processor) and `lingbot-vla-4b` (model config). RoboTwin is the `RLinf_support` branch with its assets downloaded.

`--init` lets finished background processes be reaped (the notebook launches training with `nohup setsid`). One run peaks at ~170 GB of host RAM, so the machine needs at least ~200 GB. If several participants share one machine, start each container on the default bridge network (not `--network host`) so that their Ray clusters do not collide on ports, and publish a different JupyterLab port for each.

Before the workshop, run the notebook once per machine type as a dry run to check the GPUs, paths and timings. Measured on 4 × W7900D: the first evaluation result arrived 16–18 min after launch. The later evaluations (~13 min apart) and the ~55-minute total are estimated from the measured step and evaluation times.